In [32]:
import math
import os
import pandas as pd
import numpy as np


In [33]:
#calculate passage number when I ran out of passage 5 vials ;;
def passage_number(time):
    base_vial = 6
    if time > 2:
        base_vial = base_vial-1 
    pnum = base_vial + (3 * time)
    return pnum



def well_namer(row, col):
    well_name = str(chr(ord('@')+ row)) + str(col).rjust(2, '0')  #make the number have a left align, adding a zero
    return well_name

import re

def extract_number(string):
    match = re.search(r'\d+', string)
    if match:
        return int(match.group())
    else:
        return None




In [34]:
# Get 96-well plate CSV file from benchiling/Notion NOTE: don't initialze an empty df, use a list and convert after.

plate_path = (
    "plate_metadata/"  #'/home/mattiazzilab/Documents/Allie_Scripts/May 8 seeding.csv'
)

export_path = "plate_metadata/"  #'/mnt/bigdisk1/Allie_S/Replicative_Age_Project/Data Mining/metadata/'


def load_plate_df(path):
    raw_plate_df = pd.read_csv(path, header=0, usecols=range(1, 13)).dropna(
        axis=1, how="all"
    )
    plate_df = raw_plate_df.dropna(axis=0, thresh=2)
    print(plate_df.shape)  # Checks if correct number of rows and columns

    return plate_df


def make_map_df(condition_cols, aux_cols):
    """
    Return a df with the required columns
    """
    main_columns = [
        "Metadata_Well",
        "Metadata_WellRow",
        "Metadata_WellColumn",
        "Metadata_Field",
        "Metadata_RowColFieldCode",
        "Staining",
    ]
    df_cols = main_columns + condition_cols + aux_cols
    plate_map_df = pd.DataFrame(columns=df_cols)
    print(plate_map_df.columns)
    return plate_map_df


# plate_df.head(13)
# print(columns_row)
condition_cols = [
    "SerialPassage_BatchNumber",
    "AgeGroup",
    "PassageNumber",
]
aux_cols = [
    "Drug",
    "FlaggedBatch",
    "TimepointName",
]
make_map_df(condition_cols, aux_cols)

Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'Metadata_RowColFieldCode', 'Staining',
       'SerialPassage_BatchNumber', 'AgeGroup', 'PassageNumber', 'Drug',
       'FlaggedBatch', 'TimepointName'],
      dtype='object')


,Metadata_Well,Metadata_WellRow,Metadata_WellColumn,Metadata_Field,Metadata_RowColFieldCode,Staining,SerialPassage_BatchNumber,AgeGroup,PassageNumber,Drug,FlaggedBatch,TimepointName


In [35]:
def conditional_ops(condition_cols, well_metadata, aux_cols=[]):
    updates_dict = {}
    text = well_metadata.split(" ")
    # use regex to extract the numerical bits e.g.  grab the number after the B in the serial passage batch
    all_cols = condition_cols+aux_cols
    for condition in all_cols:
        if condition == "SerialPassage_BatchNumber":
            updates_dict[condition] = text[0].split("B")[1] 
        elif condition == "TreatmentGroup":
            updates_dict[condition] = text[0]
        elif condition == "AgeGroup":
            # time used to mean age group - deprecated term but still used in code
            updates_dict[condition] = extract_number(text[1])  
        elif condition == "PassageNumber":
            # passage_number(Int(time)) - use the function if you don;t have passage number in the table
            updates_dict[condition] = extract_number(text[2]) 
        elif condition == "Drug":
            # grab drug, name an
            if ("_" in text[1]):  
                # if there is an underscore than the well is drug-treated for this group
                updates_dict[condition] = text[1].split("_")[1]  
                # grab the drug name past the underscore
            else:
                updates_dict[condition] = "None"
        elif condition == "FlaggedBatch":
            # flag passage if we have "Flagged" in the serial passage batch
            updates_dict[condition] = "Flagged" in text[0]
        elif condition == "TimepointName":
            updates_dict[condition] = text[0] + " " + text[1] + " " + text[2]
        elif condition == "ShortStaining":
            new_text = []
            for term in text:
                term = term.removesuffix("INK4A")
                term = term.removesuffix("CIP1WAF1")
                if term != "+":
                    new_text.append(term)
            stains = "_".join(new_text[len(condition_cols) : len(new_text)])
            updates_dict[condition] = stains
    return updates_dict

In [36]:
def export_platemap_csv(plate_df, plate_name, export_path, condition_cols, aux_cols=[]):
    plate_map_df = make_map_df(condition_cols,aux_cols)
    columns_row = plate_df.columns  # get_columns_row(plate_df)
    for index, data in plate_df.iterrows():
        row = data.to_list()
        for count, value in enumerate(row):
            curr_well = value
            if pd.isna(curr_well):
                continue
            row_list = []
            # get string data and label of row in df (e.g. col1, text= R1T0 EAA1-488 Tfn-647)
            # regex to separate into different variables
            # then add them to dict with their respective col index (label) and index of the row in the column (column.index)
            for i in range(40):
                row_entry = {}

                row_index = index + 1  # Make it 1-indexed

                column_index = columns_row[count]  # Use header row for column index
                # print(column_index)

                if ~np.isnan(row_index):
                    well_name = well_namer(row_index, column_index)
                else:
                    well_name = "Empty"
                    continue
                
                # information for the field is just 1-40, nothing else changes
                field = i + 1
                rowcolfield = f"r{str(row_index).rjust(2, '0')}c{str(column_index).rjust(2, '0')}f{str(field).rjust(2, '0')}"
                
                # seperate well metadata by space
                text = curr_well.split(" ")
                
                #stains come after all the conditions
                n_conditions = len(condition_cols)
                stains = " ".join(
                    text[n_conditions : len(text)]
                ) 
                
                row_entry.update(
                    {
                        "Metadata_Well": well_name,
                        "Metadata_WellRow": row_index,
                        "Metadata_WellColumn": column_index,
                        "Metadata_Field": field,
                        "Staining": stains.strip(),
                        "Metadata_RowColFieldCode":rowcolfield
                    }
                )
                conditional_entries = conditional_ops(condition_cols, curr_well, aux_cols)
                row_entry.update(conditional_entries)

                row_list.append(row_entry)

            rows = pd.DataFrame(row_list)
            plate_map_df = pd.concat([plate_map_df, rows], ignore_index=True)
            # column_index = column_index+1
    plate_map_df.to_csv(os.path.join(export_path, f"{plate_name}_map.csv"), index=False)

In [37]:

for root, dirs, files in os.walk(plate_path):
    for filename in files:
        if filename.endswith(".csv") and "map" not in filename and "Pilot" not in filename:
            file_path = os.path.join(root,filename)
            plate_df = load_plate_df(os.path.abspath(file_path))
            display(plate_df)
            export_path = os.path.abspath(root)
            print(f"Exporting {filename} to {export_path}")
            plate_name = filename.split(".")[0]
            export_platemap_csv(plate_df, plate_name, export_path, condition_cols, aux_cols=aux_cols)
            

(6, 10)


,2,3,4,5,6,7,8,9,10,11
1,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
2,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
3,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
4,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
5,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647
6,SPB1 AG4 P17 LAMP1-488 + MitoRed + WGA-CF640R,SPB2 AG3 P14 LAMP1-488 + MitoRed + WGA-CF640R,SPB3 AG2 P11 LAMP1-488 + MitoRed + WGA-CF640R,SPB4 AG1 P9 LAMP1-488 + MitoRed + WGA-CF640R,SPB5 AG0 P6 LAMP1-488 + MitoRed + WGA-CF640R,SPB1 AG4 P17 EEA1-488 + ConA-594 + Tfn-647,SPB2 AG3 P14 EEA1-488 + ConA-594 + Tfn-647,SPB3 AG2 P11 EEA1-488 + ConA-594 + Tfn-647,SPB4 AG1 P9 EEA1-488 + ConA-594 + Tfn-647,SPB5 AG0 P6 EEA1-488 + ConA-594 + Tfn-647


Exporting mar13plate.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20240313_rep01_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'Metadata_RowColFieldCode', 'Staining',
       'SerialPassage_BatchNumber', 'AgeGroup', 'PassageNumber', 'Drug',
       'FlaggedBatch', 'TimepointName'],
      dtype='object')
(8, 6)


,2,3,4,5,6,7
0,SPB2 AG5 P21 Phalloidin-647 + ConA-594,SPB3 AG4 P18 Phalloidin-647 + ConA-594,SPB4 AG3 P15 Phalloidin-647 + ConA-594,SPB5 AG2 P13 Phalloidin-647 + ConA-594,SPB6 AG1 P10 Phalloidin-647 + ConA-594,SPB7 AG0 P8 Phalloidin-647 + ConA-594
1,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
2,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
3,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
4,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
5,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
6,SPB2 AG5 P21 LAMP1-488 + MitoRed,SPB3 AG4 P18 LAMP1-488 + MitoRed,SPB4 AG3 P15 LAMP1-488 + MitoRed,SPB5 AG2 P13 LAMP1-488 + MitoRed,SPB6 AG1 P10 LAMP1-488 + MitoRed,SPB7 AG0 P8 LAMP1-488 + MitoRed
7,SPB2 AG5 P21 Phalloidin-647 + WGA-647 + ConA-594,SPB3 AG4 P18 Phalloidin-647 + WGA-647 + ConA-594,SPB4 AG3 P15 Phalloidin-647 + WGA-647 + ConA-594,SPB5 AG2 P13 Phalloidin-647 + WGA-647 + ConA-594,SPB6 AG1 P10 Phalloidin-647 + WGA-647 + ConA-594,SPB7 AG0 P8 Phalloidin-647 + WGA-647 + ConA-594


Exporting oct18plate.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20241018_rep03_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'Metadata_RowColFieldCode', 'Staining',
       'SerialPassage_BatchNumber', 'AgeGroup', 'PassageNumber', 'Drug',
       'FlaggedBatch', 'TimepointName'],
      dtype='object')
(6, 10)


...

Exporting apr10plate.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250410_rep06_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'Metadata_RowColFieldCode', 'Staining',
       'SerialPassage_BatchNumber', 'AgeGroup', 'PassageNumber', 'Drug',
       'FlaggedBatch', 'TimepointName'],
      dtype='object')
(6, 12)


,1,2,3,4,5,6,7,8,9,10,11,12
1,SPB12 AG2_Doxo P12 LAMP1-488 + MitoRed + Phall...,SPB3 AG9 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG8 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB2 AG7 P24 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG6 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG5 P20 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG3 P17 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG2 P15 LAMP1-488 + MitoRed + Phalloidin...,SPB12 AG1 P13 LAMP1-488 + MitoRed + Phalloidin...,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647
2,SPB12 AG2_Doxo P12 LAMP1-488 + MitoRed + Phall...,SPB3 AG9 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG8 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB2 AG7 P24 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG6 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG5 P20 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG3 P17 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG2 P15 LAMP1-488 + MitoRed + Phalloidin...,SPB12 AG1 P13 LAMP1-488 + MitoRed + Phalloidin...,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647
3,SPB12 AG2_Doxo P12 LAMP1-488 + MitoRed + Phall...,SPB3 AG9 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG8 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB2 AG7 P24 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG6 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG5 P20 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG3 P17 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG2 P15 LAMP1-488 + MitoRed + Phalloidin...,SPB12 AG1 P13 LAMP1-488 + MitoRed + Phalloidin...,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647
4,SPB12 AG2_Doxo P12 LAMP1-488 + MitoRed + Phall...,SPB3 AG9 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG8 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB2 AG7 P24 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG6 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG5 P20 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG3 P17 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG2 P15 LAMP1-488 + MitoRed + Phalloidin...,SPB12 AG1 P13 LAMP1-488 + MitoRed + Phalloidin...,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647
5,SPB12 AG2_Doxo P12 LAMP1-488 + MitoRed + Phall...,SPB3 AG9 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG8 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB2 AG7 P24 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG6 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG5 P20 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG3 P17 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG2 P15 LAMP1-488 + MitoRed + Phalloidin...,SPB12 AG1 P13 LAMP1-488 + MitoRed + Phalloidin...,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647
6,SPB12 AG2_Doxo P12 LAMP1-488 + MitoRed + Phall...,SPB3 AG9 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG8 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB2 AG7 P24 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG6 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB9 AG5 P20 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB10 AG3 P17 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG2 P15 LAMP1-488 + MitoRed + Phalloidin...,SPB12 AG1 P13 LAMP1-488 + MitoRed + Phalloidin...,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647,SPB13 AG0 P9 LAMP1-488 + MitoRed + Phalloidin-647


Exporting may01plate.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250501_rep07_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'Metadata_RowColFieldCode', 'Staining',
       'SerialPassage_BatchNumber', 'AgeGroup', 'PassageNumber', 'Drug',
       'FlaggedBatch', 'TimepointName'],
      dtype='object')
(6, 12)


,1,2,3,4,5,6,7,8,9,10,11,12
1,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB5_Flagged AG2 P14 LAMP1-488 + MitoRed + Pha...,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:800 + Phalloidin-647,SPB5 AG3_Doxo P16 p16INK4A-594 1:800 + Phalloi...
2,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB5_Flagged AG2 P14 LAMP1-488 + MitoRed + Pha...,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:1600 + Phalloidin-647,SPB5 AG3_Doxo P16 p16INK4A-594 1:1600 + Phallo...
3,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB5_Flagged AG2 P14 LAMP1-488 + MitoRed + Pha...,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:2600 + Phalloidin-647,SPB5 AG3_Doxo P16 p16INK4A-594 1:2600 + Phallo...
4,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB5_Flagged AG2 P14 LAMP1-488 + MitoRed + Pha...,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:3600 + Phalloidin-647,SPB5 AG3_Doxo P16 p16INK4A-594 1:3600 + Phallo...
5,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB5_Flagged AG2 P14 LAMP1-488 + MitoRed + Pha...,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p16INK4A-594 1:3600 LMNB1 1:400 +...,SPB5 AG3_Doxo P16 p16INK4A-594 1:3600 LMNB1 1:...
6,SPB8 AG3_Doxo P16 LAMP1-488 + MitoRed + Phallo...,SPB2 AG8 P29 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG7 P27 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG6 P25 LAMP1-488 + MitoRed + Phalloidin-647,SPB5 AG5 P22 LAMP1-488 + MitoRed + Phalloidin-647,SPB4 AG4 P18 LAMP1-488 + MitoRed + Phalloidin-647,SPB8 AG3 P16 LAMP1-488 + MitoRed + Phalloidin-647,SPB5_Flagged AG2 P14 LAMP1-488 + MitoRed + Pha...,SPB10 AG1 P10 LAMP1-488 + MitoRed + Phalloidin...,SPB11 AG0 P7 LAMP1-488 + MitoRed + Phalloidin-647,SPB11 AG0 P7 p21-594 1:6000 + Phalloidin-647,SPB5 AG3_Doxo P16 p21-594 1:600 + Phalloidin-647


Exporting mar28plate.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250328_rep05_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'Metadata_RowColFieldCode', 'Staining',
       'SerialPassage_BatchNumber', 'AgeGroup', 'PassageNumber', 'Drug',
       'FlaggedBatch', 'TimepointName'],
      dtype='object')


In [38]:
#for the pilot one
for root, dirs, files in os.walk(plate_path):
    for filename in files:
        if (
            filename.endswith(".csv")
            and "map" not in filename
            and "Pilot" in filename
        ):
            file_path = os.path.join(root, filename)
            plate_df = load_plate_df(os.path.abspath(file_path))
            display(plate_df)
            export_path = os.path.abspath(root)
            print(f"Exporting {filename} to {export_path}")
            plate_name = filename.split(".")[0]
            
            pilot_conditions = ["TreatmentGroup"]
            pilot_aux = ["ShortStaining"]
            
            export_platemap_csv(
                plate_df, plate_name, export_path, condition_cols=pilot_conditions, aux_cols=pilot_aux
            )


(6, 10)


,2,3,4,5,6,7,8,9,10,11
1,Doxo LMNB1,P7 LMNB1,Doxo Ki67,P7 Ki67,Doxo LMNB1 + Ki67 + DAPI,P7 LMNB1 + Ki67 + DAPI,Doxo Phalloidin,P7 Phalloidin,Doxo DAPI,P7 DAPI
2,Doxo LMNB1,P7 LMNB1,Doxo Ki67,P7 Ki67,Doxo LMNB1 + Ki67 + DAPI,P7 LMNB1 + Ki67 + DAPI,Doxo Phalloidin,P7 Phalloidin,Doxo DAPI,P7 DAPI
3,Doxo LMNB1 + p16INK4A + DAPI,P7 LMNB1 + p16INK4A + DAPI,Doxo Ki67 + p16INK4A + DAPI,P7 Ki67 + p16INK4A + DAPI,Doxo LMNB1 + Ki67 + p16INK4A + DAPI,P7 LMNB1 + Ki67 + p16INK4A + DAPI,Doxo LMNB1 + Ki67 + p16INK4A + DAPI + Phalloidin,P7 LMNB1 + Ki67 + p16INK4A + DAPI + Phalloidin,Doxo p16INK4A,P7 p16INK4A
4,Doxo LMNB1 + p16INK4A + DAPI,P7 LMNB1 + p16INK4A + DAPI,Doxo Ki67 + p16INK4A + DAPI,P7 Ki67 + p16INK4A + DAPI,Doxo LMNB1 + Ki67 + p16INK4A + DAPI,P7 LMNB1 + Ki67 + p16INK4A + DAPI,Doxo LMNB1 + Ki67 + p16INK4A + DAPI + Phalloidin,P7 LMNB1 + Ki67 + p16INK4A + DAPI + Phalloidin,Doxo p16INK4A,P7 p16INK4A
5,Doxo LMNB1 + p21CIP1WAF1 + DAPI,P7 LMNB1 + p21CIP1WAF1 + DAPI,Doxo Ki67 + p21CIP1WAF1 + DAPI,P7 Ki67 + p21CIP1WAF1 + DAPI,Doxo LMNB1 + Ki67 + p21CIP1WAF1 + DAPI,P7 LMNB1 + Ki67 + p21CIP1WAF1 + DAPI,Doxo LMNB1 + Ki67 + p21CIP1WAF1 + DAPI + Phall...,P7 LMNB1 + Ki67 + p21CIP1WAF1 + DAPI + Phalloidin,Doxo p21CIP1WAF1,P7 p21CIP1WAF1
6,Doxo LMNB1 + p21CIP1WAF1 + DAPI,P7 LMNB1 + p21CIP1WAF1 + DAPI,Doxo Ki67 + p21CIP1WAF1 + DAPI,P7 Ki67 + p21CIP1WAF1 + DAPI,Doxo LMNB1 + Ki67 + p21CIP1WAF1 + DAPI,P7 LMNB1 + Ki67 + p21CIP1WAF1 + DAPI,Doxo LMNB1 + Ki67 + p21CIP1WAF1 + DAPI + Phall...,P7 LMNB1 + Ki67 + p21CIP1WAF1 + DAPI + Phalloidin,Doxo p21CIP1WAF1,P7 p21CIP1WAF1


Exporting Pilotplate.csv to /mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20251001_pilot_metadata
Index(['Metadata_Well', 'Metadata_WellRow', 'Metadata_WellColumn',
       'Metadata_Field', 'Metadata_RowColFieldCode', 'Staining',
       'TreatmentGroup', 'ShortStaining'],
      dtype='object')
